In [0]:
# Bronze: ingest raw Prometheus range-query JSON from the landing Volume into a Delta table.
# Auto Loader tracks processed files in the checkpoint, so re-runs only pick up new files.
from pyspark.sql import functions as F

SRC = "/Volumes/homelab/landing/prom_raw/"
CKPT = "/Volumes/homelab/landing/checkpoints/bronze_prom"
TARGET = "homelab.bronze.prom_raw"

schema = """
  run_id STRING, metric_name STRING, query STRING,
  chunk_start STRING, chunk_end STRING, step_s LONG,
  response STRUCT<
    status: STRING,
    data: STRUCT<
      resultType: STRING,
      result: ARRAY<STRUCT<metric: MAP<STRING,STRING>, values: ARRAY<ARRAY<STRING>>>>
    >
  >
"""

stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("pathGlobFilter", "*.json")
    .option("recursiveFileLookup", "true")
    .schema(schema)
    .load(SRC)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingest_ts", F.current_timestamp())
)

(stream.writeStream
    .option("checkpointLocation", CKPT)
    .trigger(availableNow=True)   # process what's new, then stop
    .toTable(TARGET)
    .awaitTermination())